# 01 — Data Ingestion: Nigeria Multi-Tier Framework Energy Access Survey (2018)

This notebook documents the acquisition and first-look validation of the study data: the World Bank/ESMAP **Multi-Tier Framework (MTF) Energy Access Household Survey** for Nigeria (2018), covering 3,669 households in North-West Nigeria.

The raw data is not redistributed in this repository. It is publicly available from the [World Bank Microdata Catalog](https://microdata.worldbank.org/index.php/catalog/3865); this notebook extracts and validates a locally downloaded copy, so the full pipeline is reproducible from the original source.

## Setup

Three standard-library and scientific tools: `zipfile` for archive extraction, `pathlib` for portable file paths, and `pandas` for data handling.

In [1]:
import zipfile
from pathlib import Path
import pandas as pd

print("Toolboxes loaded.")

Toolboxes loaded.


## Extracting the raw data

Extraction is performed in code rather than manually so that every step of data preparation is documented and reproducible. Paths are relative to the repository's `notebooks/` directory.

In [2]:
RAW = Path("../raw_data")
EXTRACTED = RAW / "extracted"
EXTRACTED.mkdir(exist_ok=True)

with zipfile.ZipFile(RAW / "raw-data_nigeria.zip") as z:
    z.extractall(EXTRACTED)

print("Extraction complete.")

Extraction complete.


## Survey structure

The survey ships as Stata (`.dta`) files — the standard distribution format for World Bank and DHS microdata — with one file per questionnaire section. `pandas.read_stata()` reads them directly, preserving value labels (the human-readable answer texts).

In [3]:
dta_files = sorted(EXTRACTED.rglob("*.dta"))
for f in dta_files:
    size_kb = f.stat().st_size // 1024
    print(f"{f.name:45s} {size_kb:>8,} KB")

MTF_HH_SEC_C_BATTERY.dta                             6 KB
MTF_HH_SEC_F_FUEL.dta                              274 KB
MTF_HH_SEC_HH_ASSET_LONG.dta                       758 KB
MTF_HH_SEC_HH_ASSET_WIDE.dta                       388 KB
MTF_NG_BSS_SEC_A2.dta                            1,401 KB
MTF_NG_HH_Identification.dta                       564 KB
MTF_NG_HH_SEC_A1.dta                             2,135 KB
MTF_NG_HH_SEC_B.dta                              1,856 KB
MTF_NG_HH_SEC_C.dta                              5,480 KB
MTF_NG_HH_SEC_C_BATTERY.dta                      5,480 KB
MTF_NG_HH_SEC_C_SOLAR.dta                           27 KB
MTF_NG_HH_SEC_E.dta                              1,269 KB
MTF_NG_HH_SEC_F.dta                                550 KB
MTF_NG_HH_SEC_F_FUEL.dta                           274 KB
MTF_NG_HH_SEC_G_LAMP_CANDLE.dta                  3,908 KB
MTF_NG_HH_SEC_G_LIGHT.dta                          582 KB
MTF_NG_HH_SEC_H.dta                                402 KB
MTF_NG_HH_SEC_

The file names mirror the questionnaire's section letters (A: identification and roster; B: dwelling and financial access; C: electricity supply, including solar devices; E: willingness to pay; F: fuel-based lighting; L: consumption and expenditure). The full questionnaire and codebook are available with the dataset at the source above.

## Household identification file

One row per household, carrying the geographic and administrative identifiers — state, LGA, ward, enumeration area — and the urban/rural classification.

In [4]:
ident = pd.read_stata(next(f for f in dta_files if "Identification" in f.name))
print("Shape (rows, columns):", ident.shape)
ident.head()

Shape (rows, columns): (3669, 14)


,HH_ID,lga,EA_code,state_id,ward,Urbanization,locality,EA_name,hhid,consent,int_start_time,language,Other_language,int_end_time
0,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10001.0,YES,2017-11-20T06:47:00-05:00,Hausa,,2017-11-20T07:57:06-05:00
1,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10003.0,YES,2017-11-20T06:48:43-05:00,English,,2017-11-20T07:48:29-05:00
2,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10004.0,YES,2017-11-20T08:03:34-05:00,Hausa,,2017-11-20T09:08:46-05:00
3,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10005.0,YES,2017-11-20T06:52:21-05:00,Hausa,,2017-11-20T07:55:20-05:00
4,1.000506e+12,GUDU,100050612.0,Sokoto,BACHAKA,Rural,SALAWA,MAIGARI SAIDU,10006.0,YES,2017-11-20T06:49:41-05:00,Hausa,,2017-11-20T07:39:52-05:00


## Structure checks

Three validations before any analysis: the identifying key is unique (no duplicate households), the urban/rural split is roughly balanced (1,835 urban / 1,833 rural), and column names are confirmed against the codebook.

In [5]:
# YOUR TURN — column names
ident.columns

Index(['HH_ID', 'lga', 'EA_code', 'state_id', 'ward', 'Urbanization',
       'locality', 'EA_name', 'hhid', 'consent', 'int_start_time', 'language',
       'Other_language', 'int_end_time'],
      dtype='str')

In [6]:
# YOUR TURN — urban vs rural counts
ident['Urbanization'].value_counts(dropna=False)


Urbanization
Urban    1835
Rural    1833
NaN         1
Name: count, dtype: int64

In [7]:
# YOUR TURN — duplicate check
ident['HH_ID'].duplicated().sum()


np.int64(0)

## Current solar-device ownership

Section C includes a dedicated file for households that use solar-based devices. Its size relative to the full sample is analytically decisive, as shown below.

In [8]:
solar = pd.read_stata(next(f for f in dta_files if "SOLAR" in f.name))
print("Shape:", solar.shape)
print("Unique households owning a solar device:", solar["HH_ID"].nunique())
solar.head()

Shape: (52, 27)
Unique households owning a solar device: 43


,HH_ID,solar_id,C143,C144,C145,C146,C147,C148,C149,C150,...,C158,C158a,C159,C160,C161,C162,C163,C164,C165,C166
0,1.000506e+12,1,Module type 3W,Yes,No,Solar Home system,0.0,888,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,Yes,12,No problems
1,1.003514e+12,1,Saroda,Yes,No,Solar Home system,0.0,888,"Other, specify (in width and length)",888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,6,Breaks too often
2,1.004500e+12,1,CITIZEN POWER,No,No,Solar Home system,6.0,75,75cm x150cm or larger,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,888,No problems
3,1.013501e+12,1,pro,Yes,No,Solar Lantern,NaN,888,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,Yes,6,Battery problems
4,2.011513e+12,1,Sumking,No,No,Solar Lantern,NaN,1,Don't Know,888,...,NaN,NaN,NaN,NaN,NaN,NaN,No,No,12,No problems


## Implication for target definition

Only **43 of 3,669 households (1.2%)** currently own a solar device. Current ownership is therefore unusable as a modeling target: it is severely imbalanced, offers only 43 positive examples to learn from, and — more fundamentally — mismatches the business question, since a solar vendor seeks *future* adopters rather than current owners.

Notebook `02` addresses this by defining the target from the survey's willingness-to-pay experiment (Section E) instead.